# Exercise 1: route the tickets

**In pairs. 40 minutes, in class.** The cells marked `GIVEN` run as they are.
Every other code cell contains only **instructions, as comments**: write the
code underneath each one. Run the notebook from the top and keep the outputs:
they are what you hand in.

Everything runs on Claude Haiku 4.5. A full run costs about $0.06.

In [ ]:
# GIVEN: run this cell as it is.
# It loads the key from .env, creates the client, downloads the 120 tickets once,
# and defines the test set (TEST) and the pool your examples come from (POOL).
import csv
import datetime
import re
import urllib.request
from collections import Counter
from pathlib import Path

import anthropic
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"
PRICE_IN, PRICE_OUT = 1.00, 5.00   # $ per million tokens (Anthropic, 1 Oct 2026)

URL = ("https://github.com/TangoAI-AlbertSchool-Data-Eng-Cloud/albert-marketplace"
       "/releases/download/tickets-v1.0.0/tickets_v1.csv")
path = Path("data/tickets_v1.csv")
if not path.exists():
    path.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(URL, path)
tickets = {r["ticket_id"]: r for r in csv.DictReader(path.open(encoding="utf-8"))}

LABELS = ["billing", "delivery", "returns_refunds", "account", "seller", "product", "information", "other"]
TEST = ["T005", "T008", "T016", "T018", "T023", "T024", "T032", "T034", "T042", "T045"]
POOL = [f"T{i:03d}" for i in range(1, 61) if f"T{i:03d}" not in TEST]   # the other 50 of batch 1
print(len(tickets), "tickets | test set:", len(TEST), "| pool:", len(POOL), "| run on", datetime.date.today())

In [ ]:
# GIVEN: the plumbing. Read it once; you do not need to change it.
spent = Counter()                      # every token this notebook uses


def text_of(t):
    """A ticket as the model reads it: its subject, a blank line, its body."""
    return f"Subject: {t['subject']}\n\n{t['body']}"


def ask(user, system, max_tokens=1000, **extra):
    """One call to the model; counts the tokens it used."""
    r = client.messages.create(model=MODEL, max_tokens=max_tokens, system=system,
                               messages=[{"role": "user", "content": user}], **extra)
    spent["in"] += r.usage.input_tokens
    spent["out"] += r.usage.output_tokens
    return r


def answer_text(r):
    """The visible answer: the text blocks joined, thinking blocks skipped."""
    return "".join(b.text for b in r.content if b.type == "text")


def parse(text, labels=LABELS):
    """The LAST label word in the answer, or "" if there is none."""
    found = re.findall(r"\b(" + "|".join(labels) + r")\b", text.lower())
    return found[-1] if found else ""


def cost(tokens_in, tokens_out):
    return tokens_in / 1e6 * PRICE_IN + tokens_out / 1e6 * PRICE_OUT


def measure(name, build_prompt, system, ids=TEST):
    """Runs one prompt pattern on a list of tickets and prints the error rate with its four facts."""
    rows, tin, tout = [], 0, 0
    for tid in ids:
        t = tickets[tid]
        r = ask(build_prompt(t), system)
        tin += r.usage.input_tokens
        tout += r.usage.output_tokens
        rows.append({"ticket": tid, "gold": t["gold_label"], "second": t["ambiguous_with"],
                     "got": parse(answer_text(r)), "answer": answer_text(r)})
    strict = sum(x["got"] != x["gold"] for x in rows)
    either = sum(x["got"] not in (x["gold"], x["second"]) for x in rows)
    print(f"{name}: strict {strict}/{len(rows)}, either {either}/{len(rows)}, "
          f"{tin} in + {tout} out tokens, ${cost(tin, tout):.4f} | {MODEL}, {datetime.date.today()}")
    return {"name": name, "strict": strict, "either": either, "in": tin, "out": tout, "rows": rows}

## 1. Your first call (5 minutes)

The demo guide's cheat sheet has the shape of the call.

In [ ]:
# Send ticket T005 to the model with ONE call to client.messages.create.
#   model      -> MODEL (Claude Haiku 4.5)
#   max_tokens -> 50 is plenty for one label
#   system     -> one sentence naming the eight labels (", ".join(LABELS)) and asking for the label only
#   messages   -> a list with ONE user message whose content is text_of(tickets["T005"])

# Print the visible answer (answer_text) and what was billed (response.usage).



## 2. Zero-shot (8 minutes)

In [ ]:
# Write SYSTEM, the system prompt every pattern will share. It must:
#   - say who the model works for (Albert's Marketplace, a beauty marketplace)
#   - define each of the eight labels in one line (the lecture's table)
#   - END with the answer format: "Answer with the label only."

# Write zero_shot(t): the user message for one ticket. Zero-shot adds nothing
# to the ticket itself, so return its text (text_of).

# Measure it on the test set: measure(name, the function, the system prompt).



## 3. Five-shot (8 minutes)

In [ ]:
# Choose FIVE example ticket ids from POOL, covering five DIFFERENT labels.
# Never from TEST: a test case shown to the model as an example is no longer a test.

# This check must pass before you go on.

# Write five_shot(t): the five solved examples, each as
#   "Message:\n<its text>\nLabel: <its gold_label>"
# separated by blank lines, then the ticket to route in the same shape, ending with "Label:".

# Measure it with the same SYSTEM.



## 4. Chain-of-thought (7 minutes)

In [ ]:
# Write SYSTEM_COT: the same system prompt, but its last line asks for the reasoning
# FIRST and the label on the LAST line, as "Label: <label>". The parser reads the last
# label word, so the label must come last. (Tip: SYSTEM.replace(old_line, new_line).)

# Measure it: same tickets, same zero_shot user message, the new system prompt.



## 5. In-model reasoning against in-prompt reasoning (6 minutes)

In [ ]:
# Pick the first ticket chain-of-thought got wrong (either label), or T001 if none.

# In-prompt reasoning: the ticket with SYSTEM_COT (reasoning written in the answer).

# In-model reasoning: the same ticket, zero-shot SYSTEM, with thinking switched on:
#   thinking={"type": "enabled", "budget_tokens": 2000}
#   max_tokens must be LARGER than the budget, because thinking counts as output (use 3000).

# Print both side by side: the label each chose, the output tokens each was billed,
# then the visible reasoning and the model's thinking (the "thinking" blocks of the response).



In [ ]:
# YOUR ANSWER (one sentence, as a comment): did the two reason differently, and which would you pay for?

## 6. The comparison (6 minutes)

In [ ]:
# GIVEN: the table of your three patterns, with the four facts.
print(f"{'pattern':18} {'strict':>7} {'either':>7} {'in':>7} {'out':>6} {'per 1,000 tickets':>18}")
for r in [zs, fs, cot]:
    print(f"{r['name']:18} {r['strict']:>5}/10 {r['either']:>5}/10 {r['in']:>7} {r['out']:>6} "
          f"${cost(r['in'], r['out']) * 100:>16.2f}")
print(f"model {MODEL} | {datetime.date.today()} | test file: tickets-v1.0.0, {len(TEST)} tickets")

In [ ]:
# YOUR ANSWER (three sentences, as comments): which pattern would you deploy, what does it
# cost per 1,000 tickets, and why can ten tickets not settle it?

## 7. At home: the other tickets, with a stated confidence

In [ ]:
# Write SYSTEM_CONF: your best system prompt, asking for the label AND a confidence
# between 0 and 1, in a fixed two-line format you can parse, for example:
#   Label: <label>
#   Confidence: <a number between 0 and 1>

# Write parse_confidence(text): the number after "Confidence:", as a float, or None if absent.
# (A regular expression on text.lower() is enough.)

# Run it on every pool ticket except your five examples. For each ticket, keep:
# ticket_id, gold_label, ambiguous_with, the predicted label, the confidence, and whether
# the prediction is right (either label).

# Write them to predictions.csv (csv.DictWriter), then print the count and the errors.

